In [0]:
from pyspark.sql import functions as F

batch_id = "batch_001"

# Read approval from the persistent table.
# This notebook does not depend on earlier Python variables.
control_rows = (
    spark.table("workspace.migration_dev.batch_control")
    .filter(F.col("batch_id") == batch_id)
    .collect()
)

if len(control_rows) != 1:
    raise RuntimeError("Expected exactly one batch control record")

control = control_rows[0]

if control["batch_status"] != "APPROVED" or control["failed_checks"] != 0:
    raise RuntimeError("Batch is not approved; Silver load blocked")

metadata = (
    spark.table("workspace.migration_dev.bronze_metadata")
    .filter(F.col("batch_id") == batch_id)
)

inventory = (
    spark.table("workspace.migration_dev.bronze_file_inventory")
    .filter(F.col("batch_id") == batch_id)
)

silver = (
    metadata.alias("m")
    .join(inventory.alias("i"), ["batch_id", "file_name"], "inner")
    .select(
        F.col("batch_id"),
        F.trim(F.col("m.document_id")).alias("document_id"),
        F.trim(F.col("m.account_number")).alias("account_number"),
        F.trim(F.col("m.customer_name")).alias("customer_name"),
        F.upper(F.trim(F.col("m.document_type"))).alias("document_type"),
        F.expr("try_cast(m.statement_date AS DATE)").alias("statement_date"),
        F.col("file_name"),
        F.col("i.pdf_path"),
        F.lower(F.trim(F.col("m.source_sha256"))).alias("source_sha256"),
        F.col("i.file_size_bytes"),
        F.lit(control["validation_run_id"]).alias("validation_run_id"),
        F.current_timestamp().alias("processed_at"),
    )
)

# Guard against lost rows and duplicate keys after transformation.
expected_count = control["expected_metadata_count"]

if silver.count() != expected_count:
    raise RuntimeError("Silver row count differs from the approved batch")

for key in ["document_id", "file_name"]:
    duplicates = silver.groupBy(key).count().filter(F.col("count") > 1)
    if duplicates.limit(1).count() > 0:
        raise RuntimeError(f"Duplicate {key}; Silver load blocked")

if silver.filter(F.col("statement_date").isNull()).limit(1).count() > 0:
    raise RuntimeError("Invalid statement date; Silver load blocked")

silver.createOrReplaceTempView("current_silver_batch")

spark.sql("""
    MERGE INTO workspace.migration_dev.silver_documents AS target
    USING current_silver_batch AS source
    ON target.batch_id = source.batch_id
       AND target.document_id = source.document_id
    WHEN MATCHED
         AND target.validation_run_id <> source.validation_run_id
         THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

saved = (
    spark.table("workspace.migration_dev.silver_documents")
    .filter(F.col("batch_id") == batch_id)
)

print(f"Silver documents: {saved.count()}")
display(saved.orderBy("document_id").limit(5))

Silver documents: 100


batch_id,document_id,account_number,customer_name,document_type,statement_date,file_name,pdf_path,source_sha256,file_size_bytes,validation_run_id,processed_at
batch_001,DOC000001,TEST00000001,Sample Customer 001,ACCOUNT_STATEMENT,2026-09-30,DOC000001.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000001.pdf,12298c6adcbc1f0ffed046f3c14ab064da327b4fab519c9e25014e6a8182c431,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:06:26.583Z
batch_001,DOC000002,TEST00000002,Sample Customer 002,ACCOUNT_STATEMENT,2026-09-30,DOC000002.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000002.pdf,e45b766bfe7a3f38c7303239ffab722e244474ccb300ef84529b48197cbb83ce,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:06:26.583Z
batch_001,DOC000003,TEST00000003,Sample Customer 003,ACCOUNT_STATEMENT,2026-09-30,DOC000003.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000003.pdf,eecd9ffaf648015b1dcc0a49067b9724e9a38b609a1174a19046bf6d33a662c7,1543,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:06:26.583Z
batch_001,DOC000004,TEST00000004,Sample Customer 004,ACCOUNT_STATEMENT,2026-09-30,DOC000004.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000004.pdf,8a1b437face5c2d7feb3fbee111a1b98b3f72afe81110e92fa55279fc7e3ea76,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:06:26.583Z
batch_001,DOC000005,TEST00000005,Sample Customer 005,ACCOUNT_STATEMENT,2026-09-30,DOC000005.pdf,/Volumes/workspace/migration_dev/landing/batch_001/pdfs/DOC000005.pdf,e9907b9c7b5e0b6d0836ae782f5a5de81780eb22dbba4802102a97ed69773eb6,1542,4e66d857-a136-4b69-af88-a9acada6bd83,2026-10-14T10:06:26.583Z
